## 读取数据

In [1]:
data_path="../../data/ultrafineweb-en-part-0036-of-2048.parquet"

In [2]:
import pandas as pd

In [3]:
df=pd.read_parquet(data_path)

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 566016 entries, 0 to 566015
Data columns (total 3 columns):
 #   Column   Non-Null Count   Dtype 
---  ------   --------------   ----- 
 0   content  566016 non-null  object
 1   score    566016 non-null  object
 2   source   566016 non-null  object
dtypes: object(3)
memory usage: 13.0+ MB


In [5]:
from openai import OpenAI

In [6]:
import rich

In [7]:
import os
from dotenv import load_dotenv

In [8]:
load_dotenv("../../.env")

True

In [9]:
base_url=os.getenv("GEM_LLM_BASE_URL")
api_key=os.getenv("GEM_LLM_API_KEY")
model_name=os.getenv("GEM_LLM_MODEL_NAME")

In [10]:
client = OpenAI(
    base_url=base_url,
    api_key=api_key
)

In [11]:
def save_demo(response:str,file_name:str="demo.txt")->None:
    with open(file_name,"w") as f:
        f.write(response)

## tag_annotation

In [12]:
tag_annotation_prompt=""
with open("../../src/gem/prompts/tag_annotation.md","r") as f:
    tag_annotation_prompt=f.read()

In [13]:
def call_llm(prompt:str)->str:
    response = client.chat.completions.create(
        model=model_name,
        messages=[{"role":"user","content":prompt}],
        temperature=0.7,
        max_tokens=20480,
        top_p=0.95
    )
    return response.choices[0].message.content.rsplit("</think>",1)[-1].lstrip()

In [14]:
def tag_annotation(seed:int)->str:
    prompt=tag_annotation_prompt.replace("{text}",df.loc[seed,"content"])
    return call_llm(prompt)

In [15]:
import re
from typing import Optional
from pydantic import BaseModel

class TagAnnotation(BaseModel):
    """输出字符串的结构化数据类，对应所有提取字段"""
    multi_step: bool  # 多步标识，布尔型
    summary: str      # 摘要，字符串
    domain: str       # 领域，字符串（支持多值逗号分隔）
    platform: str     # 平台，字符串
    task: str         # 任务，字符串

In [16]:
def extract_tag_annotation(response: str) -> TagAnnotation:
    """
    解析输出字符串，提取字段并实例化TagAnnotation对象
    :param response: 待解析的标签格式字符串
    :return: TagAnnotation实例，可通过.属性访问所有字段
    """
    # 正则匹配函数：封装通用匹配逻辑，处理空白、默认值
    def _match_tag(pattern: str, default: Optional[str] = "") -> str:
        match = re.search(pattern, response, re.DOTALL)  # re.DOTALL让.匹配换行符
        return match.group(1).strip() if match else default

    # 逐个匹配各标签的内容，正则模式为<标签>(.*?)</标签>（非贪婪匹配）
    multi_step_str = _match_tag(r'<multi_step>(.*?)</multi_step>', default='False')
    summary = _match_tag(r'<summary>(.*?)</summary>')
    domain = _match_tag(r'<domain>(.*?)</domain>')
    platform = _match_tag(r'<platform>(.*?)</platform>')
    task = _match_tag(r'<task>(.*?)</task>')

    multi_step = multi_step_str.lower() == 'true'

    # 实例化数据类并返回
    return TagAnnotation(
        multi_step=multi_step,
        summary=summary,
        domain=domain,
        platform=platform,
        task=task
    )

In [17]:
response=tag_annotation(125)

In [18]:
rich.print(response)

<multi_step>True</multi_step>
<summary>User performs multi-step translation task using an online translation service</summary>
<domain>internet_and_telecom</domain>
<platform>computer</platform>
<task>language_translation</task>

In [19]:
rich.print(extract_tag_annotation(response))

TagAnnotation(
    multi_step=True,
    summary='User performs multi-step translation task using an online translation service',
    domain='internet_and_telecom',
    platform='computer',
    task='language_translation'
)

In [20]:
# for i in range(100):
#     response=tag_annotation(i)
#     ta=extract_tag_annotation(response)
#     if ta.multi_step:
#         rich.print(i)
#         rich.print(ta)
#         break

In [20]:
import json

ta=extract_tag_annotation(response)
save_demo(json.dumps(ta.model_dump(),ensure_ascii=False,indent=4),"tag_annotation_demo.txt")

## workflow_and_tool_discovery

In [21]:
workflow_and_tool_discovery_prompt=""
with open("../../src/gem/prompts/workflow_and_tool_discovery.md","r") as f:
    workflow_and_tool_discovery_prompt=f.read()

In [22]:
def workflow_and_tool_discovery(seed:int)->str:
    prompt=workflow_and_tool_discovery_prompt.replace("{text}",df.loc[seed,"content"])
    return call_llm(prompt)

In [23]:
import json

In [24]:
from pydantic import BaseModel, field_validator

class Workflow(BaseModel):
    description: str
    steps: str
    execution_graph: str
    actions: list[dict]
    tools: list[dict]

    @field_validator('steps')
    @classmethod
    def fix_newlines(cls, v: str) -> str:
        # 把所有的双换行或多换行统一缩减为单换行
        # 同时处理掉可能存在的 \\n 这种字面量字符串
        v = v.replace('\\n', '\n') 
        return re.sub(r'\n\s*\n', '\n', v).strip()

In [25]:
import dirtyjson
import json

# ------------------- 核心提取函数：extract_workflows -------------------
def extract_workflows(response: str) -> list[Workflow]:
    """
    从响应字符串中提取所有<workflow>块，解析为List[Workflow]
    :param response: 包含一个/多个<workflow>的原始响应字符串
    :return: Workflow对象列表，无匹配时返回空列表
    """
    def _match_tag(workflow_block: str, tag: str, default: str = "") -> str:
        """
        从单个workflow块中匹配指定子标签的内容，处理空白和缺失
        :param workflow_block: 单个<workflow>内的字符串
        :param tag: 子标签名（如description/steps）
        :param default: 匹配不到的默认值
        :return: 标签内的内容（已去前后空白）
        """
        pattern = re.compile(rf'<{tag}>(.*?)</{tag}>', re.DOTALL)  # re.DOTALL适配换行
        match = pattern.search(workflow_block)
        return match.group(1).strip() if match else default
    
    
    def _safe_json_loads(json_str: str, default: Optional[any] = None) -> any:
        if default is None: 
            default = []
        if not json_str: 
            return default
        try:
            # dirtyjson 比原生 json.loads 容错率高得多
            return dirtyjson.loads(json_str)
        except Exception:
            # 如果还是不行，尝试用正则强行补齐简单的括号错误（可选）
            return default
            
    workflows: List[Workflow] = []
    # 第一步：匹配所有<workflow>...</workflow>块（非贪婪匹配，获取独立的工作流字符串）
    workflow_pattern = re.compile(r'<workflow>(.*?)</workflow>', re.DOTALL | re.MULTILINE)
    workflow_blocks = workflow_pattern.findall(response)

    # 第二步：遍历每个workflow块，逐个解析内部子标签
    for block in workflow_blocks:
        # 匹配各子标签的原始字符串
        desc = _match_tag(block, "description")
        steps = _match_tag(block, "steps")
        exec_graph = _match_tag(block, "execution_graph")
        actions_str = _match_tag(block, "actions")
        tools_str = _match_tag(block, "tools")

        # 解析JSON格式的actions和tools（安全解析，失败返回空列表）
        actions = _safe_json_loads(actions_str)
        tools = _safe_json_loads(tools_str)

        # 实例化Workflow并加入列表
        workflow = Workflow(
            description=desc,
            steps=steps,
            execution_graph=exec_graph,
            actions=actions,
            tools=tools
        )
        workflows.append(workflow)

    return workflows

In [26]:
response=workflow_and_tool_discovery(125)

In [27]:
rich.print(response)

<workflow>
<description>Translation Service Workflow</description>
<steps>Step1: Input text in Original text window\nStep2: Select Indonesian to Yiddish translation direction\nStep3:
Use spellchecker\nStep4: Choose translation provider\nStep5: Press Translate\nStep6: Hit TTS Voice icon to 
listen\nStep7: Open back translation window\nStep8: Print translation</steps>
<execution_graph>(input_text)->(select_language)->(spellcheck)->(choose_provider)->(translate)->(listen_tts)->(back
_translate)->(print_output)</execution_graph>
<actions>[{"name":"input_text", "arguments": {"text": "original_indonesian_text"}},{"name":"select_language", 
"arguments": {"source": "indonesian", "target": "yiddish"}},{"name":"spellcheck", "arguments": {"text": 
"original_indonesian_text"}},{"name":"choose_provider", "arguments": {"provider_id": 
"google_translate"}},{"name":"translate", "arguments": {"text": "original_indonesian_text", "source": "indonesian",
"target": "yiddish", "provider_id": "google_translate"}},{"name":"listen_tts", "arguments": {"text": 
"translated_text", "language": "yiddish"}},{"name":"back_translate", "arguments": {"original": 
"original_indonesian_text", "translated": "translated_text", "source": "indonesian", "target": "yiddish", 
"provider_id": "google_translate"}},{"name":"print_output", "arguments": {"text": "translated_text"}}]</actions>
<tools>[{"name":"input_text","description":"Capture user input 
text","parameters":{"type":"object","properties":{"text":{"type":"string","description":"Original text to be 
translated"}},"required":["text"]},{"name":"select_language","description":"Set translation 
direction","parameters":{"type":"object","properties":{"source":{"type":"string","description":"Source language 
code (e.g. 'indonesian')"},"target":{"type":"string","description":"Target language code (e.g. 
'yiddish')"},"provider_id":{"type":"string","description":"Translation provider 
identifier"}},"required":["source","target","provider_id"]},{"name":"spellcheck","description":"Validate text 
spelling","parameters":{"type":"object","properties":{"text":{"type":"string","description":"Text to be checked for
spelling errors"}},"required":["text"]},{"name":"choose_provider","description":"Select translation 
service","parameters":{"type":"object","properties":{"provider_id":{"type":"string","description":"Unique 
identifier for translation provider"}},"required":["provider_id"]},{"name":"translate","description":"Perform 
translation","parameters":{"type":"object","properties":{"text":{"type":"string","description":"Text to 
translate"},"source":{"type":"string","description":"Source language 
code"},"target":{"type":"string","description":"Target language 
code"},"provider_id":{"type":"string","description":"Translation provider 
identifier"}},"required":["text","source","target","provider_id"]},{"name":"listen_tts","description":"Play audio 
of text","parameters":{"type":"object","properties":{"text":{"type":"string","description":"Text to be 
spoken"},"language":{"type":"string","description":"Language code for 
TTS"}},"required":["text","language"]},{"name":"back_translate","description":"Verify translation accuracy via back
translation","parameters":{"type":"object","properties":{"original":{"type":"string","description":"Original source
text"},"translated":{"type":"string","description":"Translated text to 
verify"},"source":{"type":"string","description":"Source language 
code"},"target":{"type":"string","description":"Target language 
code"},"provider_id":{"type":"string","description":"Translation provider 
identifier"}},"required":["original","translated","source","target","provider_id"]},{"name":"print_output","descrip
tion":"Generate physical copy of 
translation","parameters":{"type":"object","properties":{"text":{"type":"string","description":"Text to 
print"}},"required":["text"]}}]</tools>
</workflow>

In [28]:
with open("workflows_raw_demo.txt","w") as f:
    f.write(response)

In [40]:
wls=extract_workflows(response)

In [41]:
rich.print(wls)

[
    Workflow(
        description='Translation Service Workflow',
        steps='Step1: Input text in original text window\nStep2: Select Indonesian to Yiddish translation 
direction\nStep3: Choose translation provider\nStep4: Click Translate\nStep5: Use back translation feature\nStep6: 
Check translation quality\nStep7: Print translation',
        execution_graph='(input_text)->(select_language_direction)->(select_translation_provider)->(translate_text)
->(back_translate_text)->(check_translation_quality)->(print_translation)',
        actions=[
            {'name': 'input_text', 'arguments': AttributedDict([('text', 'Original Indonesian text')])},
            {
                'name': 'select_language_direction',
                'arguments': AttributedDict([('source_language', 'indonesian'), ('target_language', 'yiddish')])
            },
            {
                'name': 'select_translation_provider',
                'arguments': AttributedDict([('provider', 'google_translate')])
            },
            {
                'name': 'translate_text',
                'arguments': AttributedDict([('text', 'Original Indonesian text'), ('source_language', 
'indonesian'), ('target_language', 'yiddish'), ('provider', 'google_translate')])
            },
            {
                'name': 'back_translate_text',
                'arguments': AttributedDict([('translated_text', 'Translated Yiddish text'), ('provider', 
'google_translate')])
            },
            {
                'name': 'check_translation_quality',
                'arguments': AttributedDict([('original_text', 'Original Indonesian text'), ('translated_text', 
'Translated Yiddish text'), ('back_translated_text', 'Back-translated Indonesian text')])
            },
            {
                'name': 'print_translation',
                'arguments': AttributedDict([('translation', 'Translated Yiddish text')])
            }
        ],
        tools=[]
    )
]

In [110]:
json_data = json.dumps([wf.model_dump() for wf in wls], ensure_ascii=False, indent=4)

In [111]:
save_demo(json_data,"workflows_demo.txt")

In [105]:
# with open("workflows_demo.txt", "r") as f:
#     raw_data = json.load(f) 

# workflows = [Workflow.model_validate(item) for item in raw_data]
# rich.print(workflows)

## trajectory_generation

In [77]:
from pydantic import BaseModel, Field
from typing import List, Optional, Union, Dict, Any
import json

class ToolDefinition(BaseModel):
    """对应 <toolsets> 中的单个工具定义"""
    name: str
    description: str
    parameters: Dict[str, Any]
    
class ToolCall(BaseModel):
    """存储 <func> 标签内的工具调用信息"""
    name: str
    arguments: dict

class Message(BaseModel):
    """单条消息结构"""
    role: str  # 'user', 'assistant', 'tool'
    content: str
    # 只有 assistant 角色可能包含 tool_calls
    tool_calls: Optional[List[ToolCall]] = None

class Dialogue(BaseModel):
    """完整的对话"""
    system_prompt: str
    conversation: List[Message]

class Trajectory(BaseModel):
    """完整的轨迹"""
    toolsets: List[ToolDefinition]
    system_prompt: str
    conversation: List[Message]

In [78]:
import re

def extract_dialogue(response: str) -> Optional[Dialogue]:
    # 1. 提取 System Prompt
    system_match = re.search(r'<system>(.*?)</system>', response, re.DOTALL)
    system_prompt = system_match.group(1).strip() if system_match else ""

    # 2. 提取所有对话轮次 (user, assistant, tool)
    # 我们使用正则匹配所有合法的标签对
    # 注意：这里不处理嵌套，先按顺序抓取所有顶级标签
    tags_pattern = re.compile(r'<(user|assistant|tool)>(.*?)</\1>', re.DOTALL)
    all_turns = tags_pattern.findall(response)

    conversation = []

    for role, content in all_turns:
        content = content.strip()
        tool_calls = []

        # 3. 如果是 assistant，进一步解析内部的 <func> 标签
        if role == 'assistant':
            func_pattern = re.compile(r'<func>(.*?)</func>', re.DOTALL)
            funcs = func_pattern.findall(content)
            
            for f_json in funcs:
                try:
                    # 清洗 JSON 字符串（处理模型可能多出的换行）
                    clean_json = re.sub(r'[\x00-\x1F\x7F]', '', f_json.strip())
                    f_data = json.loads(clean_json)
                    tool_calls.append(ToolCall(
                        name=f_data.get("name", ""),
                        arguments=f_data.get("arguments", {})
                    ))
                except json.JSONDecodeError:
                    continue # 或者记录解析失败
            
            # 移除 content 中的 <func> 部分，只保留纯文本回复（可选）
            content = func_pattern.sub('', content).strip()

        conversation.append(Message(
            role=role,
            content=content,
            tool_calls=tool_calls if tool_calls else None
        ))

    return Dialogue(
        system_prompt=system_prompt,
        conversation=conversation
    )

In [79]:
trajectory_generation_prompt=""
with open("../../src/gem/prompts/trajectory_generation.md","r") as f:
    trajectory_generation_prompt=f.read()

In [80]:
def trajectory_generation(workflow:Workflow)->str:
    steps:str=workflow.steps
    tools:list[dict]=workflow.tools
    tools_str=json.dumps(tools,ensure_ascii=False)
    prompt = (
        trajectory_generation_prompt
        .replace("{candidate_tools}", tools_str)
        .replace("{current_task}", steps)
    )
    rich.print(f"prompt: {prompt}")
    return call_llm(prompt)

In [81]:
response=trajectory_generation(wls[0])

prompt: You are tasked with generating high-quality multi-turn dialogue trajectories based on a given text 
document. The trajectory should demonstrate an AI assistant helping users complete tasks while strictly following 
domain-specific rules and constraints.

You will be provided with:
- A list of Available Tool Candidates;
- A source text document that contains the description of the scenario and task steps;

## Completion Requirements
1. System Prompt: Extract and explicitly state ALL important domain-specific rules and constraints from the source 
text document.
Example:
<system>
You are a agent specialized in retail domain. Here are some basic rules to follow:
- An order can only be cancelled if its status is 'pending' ...
- Modify action can only be called once, and will change the order status to 'pending (items modified)' ...
- ...
</system>

2. User Task: Create natural, progressive user requests that test the system’s rule enforcement and constraint 
handling. Here are some features:
- Naturalness: Requests should reflect real-world use cases.
- Ambiguity: User requests are often incomplete, requiring the assistant to analyze or clarify them.
  Example: <user> I want to cancel order #W2575533. </user> (the user do not provide the specific reason, and the 
assistant should ask for clarification); <user> Recommend me a desktop. I often go out. </user> (the user do not 
explicitly state the attribute of the item, but the assistant should analyze and know it based on the stated 
preference)
- Consistent: User’s intention, persona, and their behavior should be consistent across the dialogue.
- Complex: The user’s request is challenging enough to test the assistant’s ability. Users can make requests that 
violate domain rules and are not allowed to alert the assistant (e.g., do not ask the model to verify the order 
status first).
  At least in one turn, the user’s request is very complex and require assistant to handle it carefully.
  Example 1:
  <user> I need to make several changes to my order #W2575533. Can I change the E-Reader to a different size, swap 
the Garden Hose color, and also update my shipping address </user>
  (Requires assistant to: check order status, verify each item can be modified, handle address change separately, 
remind about one-time modification limit)
  Example 2:
  <user> Check my tire pressures. If any of them are low, find me the nearest service station and also check if I 
have enough fuel to get there </user>
  (Requires: check tire pressure, evaluate condition, conditionally call find_nearest_shop, check fuel level, 
calculate if sufficient)
  Example 3:
  <user> I’m planning a three-day trip starting from Hangzhou, and I need help creating an itinerary. One more 
thing about the second day - I’m trying to be smart about my budget. If I end up booking a luxury hotel that costs 
800 CNY or more per night, then I need to be more careful with other expenses: my total spending on both 
restaurants (lunch and dinner) should stay under 350 CNY, both restaurants should be rated at least 4.0 stars, and 
the afternoon attraction ticket needs to be less than 120 CNY. </user>
  (Requires: check multiple constraints)

3. Assistant: Generate Responses that Demonstrate Rule Enforcement, Clear Communication, and Intelligent 
Problem-Solving:
- Reasoning and Adaptive Planning: The Assistant should reason through problem contexts and plan appropriate steps.
Sometimes users may not be able to directly provide the parameters for tool calls, and the assistant needs to 
accurately consider whether the parameter values can be obtained through other information and tools.
- Precondition Checks: Before executing tasks, the Assistant should validate any necessary preconditions (e.g., 
authenticating identity, verifying the status of an order).
- Domain Rules and Constraints: The Assistant must follow domain-specific rules at all times. Ensure the 
assistant’s tool call and response genuinely addresses those req

In [83]:
rich.print(response)

I understand you want to include a special character with an underline underneath the 'a' in "Rumah". Let me help 
you input this text using the virtual keyboard for the special character.
<func>
{"name": "input_text", "arguments": {"text": "Rumaẖ saya besar dan nyaman?", "use_virtual_keyboard": true, 
"special_characters": ["a̱"]}}
</func>

In [84]:
dialogue=extract_dialogue(response)

In [85]:
rich.print(dialogue)

Dialogue(system_prompt='', conversation=[])

In [42]:
json_data = json.dumps(dialogue.model_dump(), ensure_ascii=False, indent=4)
rich.print(json_data)

{
    "system_prompt": "",
    "conversation": []
}

In [43]:
save_demo(json_data,"dialogue_demo.txt")

## trajectory_refinement

In [44]:
def extract_trajectory(response: str) -> Trajectory:
    # 1. 提取工具集 <toolsets>
    toolsets_match = re.search(r'<toolsets>(.*?)</toolsets>', response, re.DOTALL)
    toolsets_data = []
    if toolsets_match:
        try:
            toolsets_data = json.loads(toolsets_match.group(1).strip())
        except json.JSONDecodeError:
            pass # 实际开发中可增加更强的 JSON 修复逻辑

    # 2. 提取系统提示词 <system>
    system_match = re.search(r'<system>(.*?)</system>', response, re.DOTALL)
    system_prompt = system_match.group(1).strip() if system_match else ""

    # 3. 提取对话历史 (按顺序匹配所有 user, assistant, tool 标签)
    # 使用正则表达式按顺序捕获
    tags_pattern = re.compile(r'<(user|assistant|tool)>(.*?)</\1>', re.DOTALL)
    all_turns = tags_pattern.findall(response)

    conversation = []

    for role, content in all_turns:
        content = content.strip()
        tool_calls = []

        # 3. 如果是 assistant，进一步解析内部的 <func> 标签
        if role == 'assistant':
            func_pattern = re.compile(r'<func>(.*?)</func>', re.DOTALL)
            funcs = func_pattern.findall(content)
            
            for f_json in funcs:
                try:
                    # 清洗 JSON 字符串（处理模型可能多出的换行）
                    clean_json = re.sub(r'[\x00-\x1F\x7F]', '', f_json.strip())
                    f_data = json.loads(clean_json)
                    tool_calls.append(ToolCall(
                        name=f_data.get("name", ""),
                        arguments=f_data.get("arguments", {})
                    ))
                except json.JSONDecodeError:
                    continue # 或者记录解析失败
            
            # 移除 content 中的 <func> 部分，只保留纯文本回复（可选）
            content = func_pattern.sub('', content).strip()

        conversation.append(Message(
            role=role,
            content=content,
            tool_calls=tool_calls if tool_calls else None
        ))

    return Trajectory(
        toolsets=[ToolDefinition(**t) for t in toolsets_data],
        system_prompt=system_prompt,
        conversation=conversation
    )

In [45]:
trajectory_refinement_prompt=""
with open("../../src/gem/prompts/trajectory_refinement.md","r") as f:
    trajectory_refinement_prompt=f.read()

In [46]:
def trajectory_refinement(workflow:Workflow,dialogue:Dialogue)->str:
    tools:list[dict]=workflow.tools
    tools_str=json.dumps(tools,ensure_ascii=False)
    prompt=(
        trajectory_refinement_prompt
        .replace("{tools}",tools_str)
        .replace("{our_traj}",json.dumps(dialogue.model_dump(),ensure_ascii=False))
    )
    return call_llm(prompt)

In [47]:
response=trajectory_refinement(wls[0],dialogue)

In [48]:
rich.print(response)

Thank you for that detailed information about Narita Airport! This is really helpful context. I can see that United
operates out of Terminal 1, which is good to know for your Gold status - you'll have access to the United Club 
lounge there.

The Narita Express getting you to Tokyo Station in about 53 minutes is quite convenient. Since you'll be arriving a
couple days before the conference, having that direct train connection will give you flexibility for sightseeing 
without needing to arrange private transportation.

However, I should mention that there's also **Haneda Airport (HND)** which is much closer to central Tokyo - only 
about 30 minutes by monorail or taxi. For business travelers with tight schedules, many prefer Haneda for the 
shorter commute. Would you have a preference between Narita and Haneda, or should I check flights into both 
airports?

**To move forward with concrete options, I still need:**

1. **Your departure airport** (e.g., SFO, LAX, ORD, etc.)
2. **Confirmed dates**: I'm assuming March 13-18, but please verify
3. **Cabin class preference**: Economy, premium economy, business, or first?
4. **Hotel priorities**: Should I search near the conference venue, or would you prefer a specific neighborhood 
with good dining options that can accommodate Sarah's vegetarian and gluten-free needs?

I'm being thorough here because Tokyo hotel demand during conference season is high, and I want to make sure we're 
searching with the right parameters to maximize your options. Once you provide these details, I can start 
presenting specific flight and hotel choices for your review.

In [49]:
trajectory=extract_trajectory(response)

In [50]:
rich.print(trajectory)

Trajectory(toolsets=[], system_prompt='', conversation=[])

In [51]:
json_data = json.dumps(trajectory.model_dump(), ensure_ascii=False, indent=4)
rich.print(json_data)

{
    "toolsets": [],
    "system_prompt": "",
    "conversation": []
}

In [52]:
save_demo(json_data,"trajectory_demo.txt")

## hallucination_detection

In [53]:
from pydantic import BaseModel, Field, field_validator

class EvaluationResult(BaseModel):
    """轨迹评估结果类"""
    R1: int = Field(description="工具调用幻觉得分 (0 或 1)")
    R2: int = Field(description="能力幻觉得分 (0 或 1)")
    R3: int = Field(description="上下文幻觉得分 (0 或 1)")

    @field_validator('R1', 'R2', 'R3')
    @classmethod
    def check_binary(cls, v: int) -> int:
        """确保得分只能是 0 或 1"""
        if v not in (0, 1):
            raise ValueError("Score must be 0 or 1")
        return v

In [54]:
import re
import json
from typing import Optional

def extract_evaluation(response: str) -> Optional[EvaluationResult]:
    """
    从模型响应中提取 JSON 评估结果。
    支持处理带有 Markdown 标签的 JSON 或纯 JSON 字符串。
    """
    # 1. 尝试寻找最外层的 JSON 花括号
    # 这样即使模型输出了 "Here is the result: { ... }" 也能成功提取
    json_pattern = re.compile(r'(\{.*?\})', re.DOTALL)
    match = json_pattern.search(response)
    
    if not match:
        print("Error: No JSON object found in response.")
        return None
    
    try:
        # 2. 清理可能存在的不可见字符并解析
        clean_json_str = re.sub(r'[\x00-\x1F\x7F]', '', match.group(1).strip())
        data = json.loads(clean_json_str)
        
        # 3. 实例化 Pydantic 模型（自动执行校验）
        return EvaluationResult.model_validate(data)
        
    except (json.JSONDecodeError, ValueError) as e:
        print(f"Extraction Failed: {e}")
        return None

In [55]:
hallucination_detection_prompt=""
with open("../../src/gem/prompts/hallucination_detection.md","r") as f:
    hallucination_detection_prompt=f.read()

In [56]:
def hallucination_detection(trajectory:Trajectory)->str:
    prompt=(
        hallucination_detection_prompt
        .replace("{trajectory}",trajectory.model_dump_json())
    )
    return call_llm(prompt)

In [57]:
response=hallucination_detection(trajectory)

In [58]:
rich.print(response)

{
  "R1": 1,
  "R2": 1,
  "R3": 1
}

In [59]:
evaluation=extract_evaluation(response)

In [60]:
rich.print(evaluation)

EvaluationResult(R1=1, R2=1, R3=1)

In [61]:
json_data=json.dumps(evaluation.model_dump(),ensure_ascii=False,indent=4)

In [62]:
save_demo(json_data,"evaluation_demo.txt")

In [63]:
# TODO: 只有R1,R2,R3都为1的数据，才被保留下来，等待后续转换格式，用于强化学习或监督微调

In [64]:
# TODO: 把保存下来的数据转为openai messages格式
# 转换的逻辑参考hardtry/utils/convert_hardgen_to_messages.py
# 最后的数据只包含system|user|assistant role